This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [1]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

I0000 00:00:1778143904.892699   84643 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model_to_test = 'jettag-hgq2'

model_configs = [
    # AdaptiveHP-models
#    {
#        "description": "AdaptiveHP acc=0.6144 ebops=266",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.6144_ebops=266(0.7reportedACC).keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "VU_DA_bitfile",
#    },
#    {
#        "description": "AdaptiveHP acc=0.7432 ebops=939",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7432_ebops=939.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "VU_DA_bitfile",
#    },
#    {
#        "description": "AdaptiveHP acc=0.7576 ebops=4453",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7576_ebops=4453.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7576_ebops=4453_VU_DA_bitfile",
#    },
    # FixedHP-models
    {
        "description": "FixedHP acc=0.7171 ebops=1353",
        "model_revision": "Training_FixedHP",
        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7171_ebops=1353.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7171_ebops=1353_VU_DA_bitfile",
    },
    # Old FixedHP (?)
#    {
#        "description": "FixedHP acc=0.7611 ebops=8919",
#        "model_revision": "Training_FixedHP",
#        "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7611_ebops=8919.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7611_ebops=8919_VU_DA_bitfile",
#    },
]

In [3]:
# Load dataset which is preprocessed in another notebook
X_train_val = np.load('Data/x_train_val.npy')
X_test = np.load('Data/x_test.npy')
y_train_val = np.load('Data/y_train_val.npy')
y_test = np.load('Data/y_test.npy')
#classes = np.load('Data/classes.npy', allow_pickle=True)


In [4]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [5]:
from keras.models import load_model
import hgq.layers
import hls4ml

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
    )
    hls_model.compile()
    return hls_model

In [ ]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")



Processing model 1/1: FixedHP acc=0.7171 ebops=1353

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu May  7 08:51:51 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7171_ebops=1353_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7171_ebops=1353_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/work/development/jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_acc=0.7171_ebops=1353_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow

In [ ]:

# Print timing summary
print(f"\n{'='*60}")
print("TIMING SUMMARY")
print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")

total_time = 0
for i, timing in enumerate(process_model_durations):
    print(f"\nModel {i+1}: {timing['model']}")
    print(f"  Time: {timing['time_seconds']:.2f} seconds ({timing['time_minutes']:.2f} minutes)")
    total_time += timing['time_seconds']

print(f"\n\n")
print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {total_time/len(process_model_durations):.2f} seconds ({total_time/len(process_model_durations)/60:.2f} minutes)")
